#### crop cells from ELDDs dataset . also add the clean label

In [1]:
# !pip install roboflow
import cv2
import os
import shutil
import yaml
import numpy as np
from ultralytics import YOLO


In [1]:
from roboflow import Roboflow
rf = Roboflow(api_key="60q1gKdq4vPi4tRgd831")
project = rf.workspace("digitizo").project("eldds1400c5-dataset-afoaf")
version = project.version(3)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ELDDS1400c5-dataset-3 in yolov8:: 100%|██████████| 2768/2768 [00:01<00:00, 1493.20it/s]


In [ ]:

# ================= CONFIGURATION =================
# 1. PATHS
IMG_DIR = r"ELDDS1400c5-dataset-3\valid\images"           # Where your full images are
RF_LABEL_DIR = r"ELDDS1400c5-dataset-3\valid\labels"      # Your Roboflow Defect Labels (txt)
OUTPUT_DIR = r"dataset_cropped_fixed_cells_clean\valid"   # Where to save the final crops

# 2. YOUR MODELS
CELL_MODEL_PATH = r"C:\Users\Rowan\Documents\Final_demo\Final_demo\demo\demo\demo\demo\src\main\backend\models/best2.pt" 

# 3. MAPPING ROBOFLOW CLASSES
data_yaml_path = r'ELDDS1400c5-dataset-3\data.yaml'

# Load class names dynamically
with open(data_yaml_path, "r") as f:
    data = yaml.safe_load(f)

DEFECT_NAMES = dict(enumerate(data["names"]))

# 4. OVERLAP THRESHOLD (Sensitivity)
IOU_THRESH = 0.1 

# ================= HELPER FUNCTIONS =================
def get_iou(boxA, boxB):
    """Calculate Intersection over Union (IoU) between two boxes"""
    # box = [x1, y1, x2, y2]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    # Use Intersection / DefectArea to check if defect is INSIDE the cell
    if boxBArea == 0: 
        return 0
    return interArea / float(boxBArea)

def convert_to_cell_coordinates(original_box, cell_box, cell_width, cell_height):
    """
    Convert defect coordinates from original image to cell-relative coordinates
    
    Parameters:
    - original_box: [x1, y1, x2, y2] in original image coordinates
    - cell_box: [cx1, cy1, cx2, cy2] in original image coordinates
    - cell_width, cell_height: dimensions of the cropped cell
    
    Returns:
    - YOLO normalized coordinates: [class_id, center_x, center_y, width, height]
    """
    # Get intersection of defect with cell
    inter_x1 = max(original_box[0], cell_box[0])
    inter_y1 = max(original_box[1], cell_box[1])
    inter_x2 = min(original_box[2], cell_box[2])
    inter_y2 = min(original_box[3], cell_box[3])
    
    # If no intersection, return None
    if inter_x1 >= inter_x2 or inter_y1 >= inter_y2:
        return None
    
    # Convert to cell-relative coordinates
    rel_x1 = (inter_x1 - cell_box[0]) / cell_width
    rel_y1 = (inter_y1 - cell_box[1]) / cell_height
    rel_x2 = (inter_x2 - cell_box[0]) / cell_width
    rel_y2 = (inter_y2 - cell_box[1]) / cell_height
    
    # Convert to YOLO format (center_x, center_y, width, height)
    center_x = (rel_x1 + rel_x2) / 2
    center_y = (rel_y1 + rel_y2) / 2
    width = rel_x2 - rel_x1
    height = rel_y2 - rel_y1
    
    return [center_x, center_y, width, height]

# ================= MAIN EXECUTION =================
# 1. Setup Folders
if os.path.exists(OUTPUT_DIR): 
    shutil.rmtree(OUTPUT_DIR)

# Create main output directories
os.makedirs(os.path.join(OUTPUT_DIR, "clean"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "defects"), exist_ok=True)

# Create subdirectories for each defect type within defects folder
for name in DEFECT_NAMES.values():
    os.makedirs(os.path.join(OUTPUT_DIR, "defects", name), exist_ok=True)

# Also create unknown_defect folder for any unexpected defect types
os.makedirs(os.path.join(OUTPUT_DIR, "defects", "unknown_defect"), exist_ok=True)

# 2. Load Cell Detector
model = YOLO(CELL_MODEL_PATH)

print(f"🚀 Starting Cell Classification Process on {IMG_DIR}...")

# Counters for statistics
clean_count = 0
defect_count = 0
multi_defect_count = 0
total_defects_found = 0

for filename in os.listdir(IMG_DIR):
    if not filename.endswith(('.jpg', '.png', '.jpeg')): 
        continue
    
    img_path = os.path.join(IMG_DIR, filename)
    label_path = os.path.join(RF_LABEL_DIR, filename.replace(os.path.splitext(filename)[1], '.txt'))
    
    img = cv2.imread(img_path)
    h_img, w_img, _ = img.shape
    
    # A. GET GROUND TRUTH DEFECTS (From Roboflow)
    defects = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:  # Skip empty lines
                    continue
                cls_id = int(parts[0])
                # Convert normalized YOLO format to pixels [x1, y1, x2, y2]
                cx, cy, w, h = map(float, parts[1:5])
                x1 = int((cx - w/2) * w_img)
                y1 = int((cy - h/2) * h_img)
                x2 = int((cx + w/2) * w_img)
                y2 = int((cy + h/2) * h_img)
                defects.append({'id': cls_id, 'box': [x1, y1, x2, y2]})

    # B. DETECT CELLS (From Your Model)
    results = model(img_path, verbose=False)[0]
    
    for i, cell_box_obj in enumerate(results.boxes):
        # Get Cell Coordinates
        cx1, cy1, cx2, cy2 = map(int, cell_box_obj.xyxy[0].cpu().numpy())
        
        cx1 = max(0, cx1)
        cy1 = max(0, cy1)
        cx2 = min(w_img, cx2)
        cy2 = min(h_img, cy2)

        cell_box = [cx1, cy1, cx2, cy2]
        cell_width = cx2 - cx1
        cell_height = cy2 - cy1
        
        if cell_width <= 0 or cell_height <= 0:
            continue

        # C. CHECK FOR ALL DEFECTS IN THIS CELL
        cell_defects = []  # Store all defects found in this cell
        defect_types = set()  # Track unique defect types
        
        for defect in defects:
            # Check if this defect overlaps with the cell
            overlap = get_iou(cell_box, defect['box'])
            
            if overlap > IOU_THRESH:
                # This defect is in the cell - add it to the list
                cell_defects.append(defect)
                defect_types.add(defect['id'])
        
        # D. CROP THE CELL
        crop = img[cy1:cy2, cx1:cx2]
        
        if crop.size == 0: 
            continue  # Skip empty crops
        
        base_name = f"{os.path.splitext(filename)[0]}_cell_{i}"
        save_name = f"{base_name}.jpg"
        
        # E. DETERMINE CELL CLASSIFICATION AND SAVE
        if len(cell_defects) == 0:
            # No defects found - this is a CLEAN cell
            save_path = os.path.join(OUTPUT_DIR, "clean", save_name)
            cv2.imwrite(save_path, crop)
            clean_count += 1
        else:
            # Cell has defects
            total_defects_found += len(cell_defects)
            
            if len(cell_defects) > 1:
                multi_defect_count += 1
            
            # Save image in appropriate defect folder
            # If only one type of defect, use that folder
            if len(defect_types) == 1:
                defect_id = next(iter(defect_types))
                if defect_id in DEFECT_NAMES:
                    folder_name = DEFECT_NAMES[defect_id]
                else:
                    folder_name = "unknown_defect"
            else:
                # Multiple defect types - save in first defect's folder
                defect_id = cell_defects[0]['id']
                if defect_id in DEFECT_NAMES:
                    folder_name = DEFECT_NAMES[defect_id]
                else:
                    folder_name = "unknown_defect"
            
            save_path = os.path.join(OUTPUT_DIR, "defects", folder_name, save_name)
            cv2.imwrite(save_path, crop)
            defect_count += 1
            
            # F. CREATE LABEL FILE FOR THIS CELL
            label_save_name = f"{base_name}.txt"
            label_save_path = os.path.join(OUTPUT_DIR, "defects", folder_name, label_save_name)
            
            with open(label_save_path, 'w') as f_label:
                for defect in cell_defects:
                    # Convert defect coordinates to cell-relative coordinates
                    rel_coords = convert_to_cell_coordinates(
                        defect['box'], 
                        cell_box, 
                        cell_width, 
                        cell_height
                    )
                    
                    if rel_coords:  # Only write if there's actual overlap
                        # Write YOLO format: class_id center_x center_y width height
                        line = f"{defect['id']} " + " ".join([f"{coord:.6f}" for coord in rel_coords])
                        f_label.write(line + "\n")

print("\n" + "="*50)
print("✅ PROCESSING COMPLETE!")
print("="*50)
print(f"Total cells processed: {clean_count + defect_count}")
print(f"  Clean cells: {clean_count}")
print(f"  Defective cells: {defect_count}")
print(f"    Cells with single defect: {defect_count - multi_defect_count}")
print(f"    Cells with multiple defects: {multi_defect_count}")
print(f"Total defects found across all cells: {total_defects_found}")
print("\n📁 Output structure:")
print(f"{OUTPUT_DIR}/")
print(f"├── clean/")
print(f"│   └── *.jpg (no label files)")
print(f"└── defects/")
print(f"    ├── {list(DEFECT_NAMES.values())[0]}/")
print(f"    │   ├── *.jpg")
print(f"    │   └── *.txt (YOLO format labels for each cell)")
print(f"    ├── ... (other defect folders)")
print(f"    └── unknown_defect/")
print("\n📝 Label files contain ALL defects found in each cell in YOLO format.")
print("   Each line: <class_id> <center_x> <center_y> <width> <height>")
print("   Coordinates are normalized relative to the cell dimensions (0-1 range).")
print("="*50)

🚀 Starting Cell Classification Process on ELDDS1400c5-dataset-3\valid\images...

✅ PROCESSING COMPLETE!
Total cells processed: 31429
  Clean cells: 25570
  Defective cells: 5859
    Cells with single defect: 5206
    Cells with multiple defects: 653
Total defects found across all cells: 6594

📁 Output structure:
dataset_cropped_fixed_cells_clean\valid/
├── clean/
│   └── *.jpg (no label files)
└── defects/
    ├── 0 Examined/
    │   ├── *.jpg
    │   └── *.txt (YOLO format labels for each cell)
    ├── ... (other defect folders)
    └── unknown_defect/

📝 Label files contain ALL defects found in each cell in YOLO format.
   Each line: <class_id> <center_x> <center_y> <width> <height>
   Coordinates are normalized relative to the cell dimensions (0-1 range).


In [11]:
results[0].boxes

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
conf: tensor([0.9171, 0.9164, 0.9140, 0.9108, 0.8915, 0.8890, 0.8814, 0.8798, 0.8768, 0.8717, 0.8714, 0.8713, 0.8692, 0.8683, 0.8672, 0.8666, 0.8651, 0.8647, 0.8633, 0.8630, 0.8614, 0.8590, 0.8571, 0.8561, 0.8554, 0.8546, 0.8529, 0.8485, 0.8480, 0.8477, 0.8468, 0.8455, 0.8446, 0.8431, 0.8429, 0.8427, 0.8425, 0.8411, 0.8405,
        0.8401, 0.8395, 0.8376, 0.8344, 0.8323, 0.8314, 0.8302, 0.8245, 0.8241, 0.8222, 0.8212, 0.8201, 0.8184, 0.8178, 0.8176, 0.8174, 0.8171, 0.8159, 0.8154, 0.8143, 0.8142, 0.8112, 0.8086, 0.8065, 0.8019, 0.7987, 0.7983, 0.7969, 0.7963, 0.7925, 0.7901, 0.7805, 0.7775

In [6]:
import os
import cv2
import supervision as sv
from ultralytics import YOLO
import numpy as np

# 1. Configuration
IMAGE_DIR = r"C:\Users\Rowan\Documents\Rowan\Yolo_test\ELDDS1400c5-dataset-3\valid/images"      # Your Roboflow images folder
LABEL_DIR = r"C:\Users\Rowan\Documents\Rowan\Yolo_test\ELDDS1400c5-dataset-3\valid/labels"      # Your Roboflow labels folder
OUTPUT_DIR = "cell_level_dataset"
# CELL_MODEL_PATH = "cell_detector.pt" # Your local model

# Load Model and Define Output Paths
model = YOLO(CELL_MODEL_PATH)
os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels", exist_ok=True)


def load_yolo_labels(label_path, img_width, img_height):
    """
    Reads a YOLO .txt file and returns a supervision.Detections object.
    Returns empty Detections if file doesn't exist.
    """
    if not os.path.exists(label_path):
        return sv.Detections.empty()

    xyxy = []
    class_ids = []

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            
            cls_id = int(parts[0])
            cx, cy, w, h = map(float, parts[1:5])

            # Convert Normalized XYWH -> Absolute XYXY
            x1 = int((cx - w / 2) * img_width)
            y1 = int((cy - h / 2) * img_height)
            x2 = int((cx + w / 2) * img_width)
            y2 = int((cy + h / 2) * img_height)

            xyxy.append([x1, y1, x2, y2])
            class_ids.append(cls_id)

    if not xyxy:
        return sv.Detections.empty()

    return sv.Detections(
        xyxy=np.array(xyxy),
        class_id=np.array(class_ids)
    )

# 2. Process each image in the dataset
for filename in os.listdir(IMAGE_DIR):
    if not filename.endswith(('.jpg', '.png', '.jpeg')): continue
    
    # Path setup
    img_path = os.path.join(IMAGE_DIR, filename)
    label_path = os.path.join(LABEL_DIR, filename.rsplit('.', 1)[0] + ".txt")
    
    if not os.path.exists(label_path): continue

    # Load image and existing defect labels
    image = cv2.imread(img_path)
    h_orig, w_orig, _ = image.shape
    

    # Load global defect annotations (YOLO format)
    defect_detections = load_yolo_labels(label_path, w_orig, h_orig)
    
    # 3. Detect Cells (Crops)
    results = model.predict(image, conf=0.5)
    cells = sv.Detections.from_ultralytics(results[0])

    # 4. Crop and Shift Coordinates
    for i, cell_box in enumerate(cells.xyxy):
        x1, y1, x2, y2 = cell_box.astype(int)
        cell_crop = image[y1:y2, x1:x2]
        
        # Filter defects inside this cell
        mask = (defect_detections.xyxy[:, 0] >= x1) & \
               (defect_detections.xyxy[:, 1] >= y1) & \
               (defect_detections.xyxy[:, 2] <= x2) & \
               (defect_detections.xyxy[:, 3] <= y2)
        
        cell_defects = defect_detections[mask]
        
        # SHIFT COORDINATES: Subtract crop top-left (x1, y1)
        relative_xyxy = cell_defects.xyxy.copy()
        relative_xyxy[:, [0, 2]] -= x1  # X shift
        relative_xyxy[:, [1, 3]] -= y1  # Y shift
        
        # 5. Save Crop and Normalized Labels
        crop_name = f"{filename.split('.')[0]}_cell_{i}"
        
        crop_h, crop_w, _ = cell_crop.shape
        
        # Avoid saving empty crops if no cells were detected properly
        if crop_h == 0 or crop_w == 0: continue
        
        cv2.imwrite(f"{OUTPUT_DIR}/images/{crop_name}.jpg", cell_crop)
        
        # Prepare label file path
        new_label_path = f"{OUTPUT_DIR}/labels/{crop_name}.txt"
        
        with open(new_label_path, "w") as f:
            for defect_idx in range(len(cell_defects)):
                # Get shifted pixel coordinates for this specific defect
                dx1, dy1, dx2, dy2 = relative_xyxy[defect_idx]
                class_id = cell_defects.class_id[defect_idx]
                
                # Clip coordinates to ensure they stay within the crop boundary
                dx1 = max(0, min(dx1, crop_w))
                dx2 = max(0, min(dx2, crop_w))
                dy1 = max(0, min(dy1, crop_h))
                dy2 = max(0, min(dy2, crop_h))

                # Calculate YOLO format: normalized center_x, center_y, width, height
                width = (dx2 - dx1)
                height = (dy2 - dy1)
                x_center = dx1 + (width / 2)
                y_center = dy1 + (height / 2)
                
                # Normalize by crop dimensions
                norm_x = x_center / crop_w
                norm_y = y_center / crop_h
                norm_w = width / crop_w
                norm_h = height / crop_h
                
                # Write to file: class_id x_center y_center width height
                f.write(f"{class_id} {norm_x:.6f} {norm_y:.6f} {norm_w:.6f} {norm_h:.6f}\n")

print(f"Processing complete. Cell-level dataset saved to: {OUTPUT_DIR}")


0: 448x640 176 cellss, 29.3ms
Speed: 9.9ms preprocess, 29.3ms inference, 3.2ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 183 cellss, 21.9ms
Speed: 3.3ms preprocess, 21.9ms inference, 1.7ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 171 cellss, 21.5ms
Speed: 2.4ms preprocess, 21.5ms inference, 1.7ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 185 cellss, 21.6ms
Speed: 2.3ms preprocess, 21.6ms inference, 1.7ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 208 cellss, 30.3ms
Speed: 2.9ms preprocess, 30.3ms inference, 3.0ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 171 cellss, 35.7ms
Speed: 3.9ms preprocess, 35.7ms inference, 6.5ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 173 cellss, 38.3ms
Speed: 7.9ms preprocess, 38.3ms inference, 1.8ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 223 cellss, 34.7ms
Speed: 9.2ms preprocess, 34.7ms inference, 1.8ms postprocess per image at

In [13]:
def load_yolo_labels(label_path, img_w, img_h):
    boxes = []
    with open(label_path) as f:
        for line in f:
            cls, xc, yc, w, h = map(float, line.split())

            x1 = (xc - w/2) * img_w
            y1 = (yc - h/2) * img_h
            x2 = (xc + w/2) * img_w
            y2 = (yc + h/2) * img_h

            boxes.append((int(cls), x1, y1, x2, y2))
    return boxes


def intersect_box(box, crop):
    # box: (x1,y1,x2,y2)
    # crop: (cx1,cy1,cx2,cy2)
    ix1 = max(box[0], crop[0])
    iy1 = max(box[1], crop[1])
    ix2 = min(box[2], crop[2])
    iy2 = min(box[3], crop[3])

    if ix1 >= ix2 or iy1 >= iy2:
        return None

    return ix1, iy1, ix2, iy2

In [14]:
import cv2
import os
from ultralytics import YOLO

# ---------- Inputs ----------
image_path = r"C:\Users\Rowan\Documents\Rowan\Yolo_test\ELDDS1400c5-dataset-3\valid\images\902_907_D211AABB3A20210257_png.rf.61edd48c4fc9ab599ee4b0891e18b2f1.jpg"
label_path = r"C:\Users\Rowan\Documents\Rowan\Yolo_test\ELDDS1400c5-dataset-3\valid\labels\902_907_D211AABB3A20210257_png.rf.61edd48c4fc9ab599ee4b0891e18b2f1.txt"     # original defect labels
out_dir = "cells"
os.makedirs(out_dir, exist_ok=True)

# ---------- Load ----------
img = cv2.imread(image_path)
H, W = img.shape[:2]

pv_labels = load_yolo_labels(label_path, W, H)

# ---------- YOLO inference ----------
results = model.predict(image_path, conf=0.25)
boxes = results[0].boxes

cell_boxes = boxes.xyxy.cpu().numpy()

# ---------- Process each cell ----------
for i, (cx1, cy1, cx2, cy2) in enumerate(cell_boxes):
    cx1, cy1, cx2, cy2 = map(int, [cx1, cy1, cx2, cy2])

    crop = img[cy1:cy2, cx1:cx2]
    ch, cw = crop.shape[:2]

    cell_labels = []

    for cls, x1, y1, x2, y2 in pv_labels:
        inter = intersect_box((x1,y1,x2,y2), (cx1,cy1,cx2,cy2))
        if inter is None:
            continue

        ix1, iy1, ix2, iy2 = inter

        # shift to crop space
        ix1 -= cx1
        iy1 -= cy1
        ix2 -= cx1
        iy2 -= cy1

        # normalize
        xc = ((ix1 + ix2) / 2) / cw
        yc = ((iy1 + iy2) / 2) / ch
        w  = (ix2 - ix1) / cw
        h  = (iy2 - iy1) / ch

        cell_labels.append(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

    # ---------- Save ----------
    cv2.imwrite(f"{out_dir}/cell_{i:03}.jpg", crop)

    with open(f"{out_dir}/cell_{i:03}.txt", "w") as f:
        f.write("\n".join(cell_labels))

image 1/1 C:\Users\Rowan\Documents\Rowan\Yolo_test\ELDDS1400c5-dataset-3\valid\images\902_907_D211AABB3A20210257_png.rf.61edd48c4fc9ab599ee4b0891e18b2f1.jpg: 448x640 215 cellss, 48.7ms
Speed: 2.7ms preprocess, 48.7ms inference, 2.9ms postprocess per image at shape (1, 3, 448, 640)


In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="60q1gKdq4vPi4tRgd831")
project = rf.workspace("digitizo").project("crop-lo56h-7fzwk")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


In [3]:
dataset_path = dataset.location
yaml_file = f"{dataset_path}/data.yaml"


model = YOLO('yolov8m.pt')

model.train(
    data=yaml_file,
    epochs=15,
    device=0,
    batch=16,
    workers=2,
    imgsz=640
)

New https://pypi.org/project/ultralytics/8.3.241 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.238  Python-3.10.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Rowan\Documents\Rowan\Yolo_test\crop-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001F506E6DF00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [20]:
import torch
torch.cuda.empty_cache()

In [ ]:
detect/train is for cell crop detector

In [ ]:
dataset_path = dataset.location
yaml_file = f"{dataset_path}/data.yaml"


model = YOLO('yolo11m.pt')

model.train(
    data=yaml_file,
    epochs=10,
    device=0,
    imgsz= 640
)